# 06 — Holdout replication and the walk-forward variant

Everything in notebooks 01–05 was scored on 2022–2024. This notebook scores two things on
**2025-01-01 → 2026-08-31**, a window nobody in this project had downloaded when the
analysis plan was written:

- **Arm A** — Study 1's strategy, unchanged, with the sizing hedge re-fit on all data through
  2024-12-31. A replication.
- **Arm B** — the walk-forward variant: annual refit of the sizing hedge on an anchored
  expanding window, and the signal window set from the fitted spread's half-life by
  `clip(round(2 × HL), 20, 250)`. This is the textbook response to Study 1's diagnosis
  (nothing persists over three years; the 60-day window is shorter than most half-lives).
  It is also run on 2022–2024, where it is reported as a *variant* of Study 1, never in
  its place.

The plan — arms, endpoints, rules, and the power statement — is `PREREGISTRATION.md`,
committed before the download. This notebook reads `reports/results/` (Study 1 and the
Arm B variant) and `reports/results_holdout/` (both arms on the holdout) and computes
nothing that is not a function of those CSVs.

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats

from pairs_teardown.config import load_config
from pairs_teardown.data.clean import align_prices, handle_missing, to_log_prices
from pairs_teardown.data.loaders import load_or_download
from pairs_teardown.stats.cointegration import estimate_hedge_ratio
from pairs_teardown.stats.inference import expected_max_sharpe, sharpe_null_sd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
cfg1 = load_config(ROOT / "configs" / "pairs.yaml")
cfgH = load_config(ROOT / "configs" / "pairs_holdout.yaml")
PAIRS = [p.name for p in cfg1.pairs]

R1 = ROOT / cfg1.output.results_dir
RH = ROOT / cfgH.output.results_dir
if not (RH / "metrics.csv").exists():
    raise FileNotFoundError(
        f"{RH / 'metrics.csv'} not found — the holdout has not been run. "
        "Read PREREGISTRATION.md, then `make run-holdout` (once)."
    )

def read(root, name):
    return pd.read_csv(root / name)

m1, i1 = read(R1, "metrics.csv"), read(R1, "inference.csv")
m1b, i1b, s1b = (read(R1, "metrics_walk_forward.csv"), read(R1, "inference_walk_forward.csv"),
                 read(R1, "walk_forward_segments.csv"))
mH, iH = read(RH, "metrics.csv"), read(RH, "inference.csv")
mHb, iHb, sHb = (read(RH, "metrics_walk_forward.csv"), read(RH, "inference_walk_forward.csv"),
                 read(RH, "walk_forward_segments.csv"))
man1 = json.loads((R1 / "run_manifest.json").read_text())
manH = json.loads((RH / "run_manifest.json").read_text())

def oos_net(metrics, value="total_return"):
    v = metrics[(metrics.basis == "net") & (metrics.period == "out_of_sample")]
    return v.set_index("pair")[value].reindex(PAIRS)

def oos_inf(inference):
    v = inference[(inference.basis == "net") & (inference.period == "out_of_sample")]
    return v.set_index("pair").reindex(PAIRS)

pd.set_option("display.precision", 3)
print(f"Study 1 window : {man1['in_sample_end']} -> {man1['data_end']}   (run {man1['run_utc'][:10]})")
print(f"Holdout window : {manH['in_sample_end']} -> {manH['data_end']}   (run {manH['run_utc'][:10]})")
print(f"walk-forward   : {manH['walk_forward']}")

## 1. Data integrity: is the past still the past?

The holdout download fetched 2015–2026 in one call. Dividends paid after 2024-12-31
back-adjust every earlier price by a constant, which shifts *log* prices by a constant and
leaves OLS slopes and daily returns unchanged — so the sizing hedge ratios fit on 2015–2021
from the new file must reproduce Study 1's manifest, and the 2015–2024 daily log returns
must match the old file. If they do not, Yahoo revised history, and that is reported here
rather than absorbed.

In [ ]:
old = load_or_download(list(cfg1.tickers), cfg1.data.start, cfg1.data.end, ROOT / cfg1.data.cache_dir)
new = load_or_download(list(cfgH.tickers), cfgH.data.start, cfgH.data.end, ROOT / cfgH.data.cache_dir)

rows = []
for p in cfg1.pairs:
    lg_new = to_log_prices(align_prices(handle_missing(new[[p.a, p.b]])))
    is_m = cfg1.split.is_mask(lg_new.index)
    g_new = estimate_hedge_ratio(pd.Series(lg_new[p.a][is_m]), pd.Series(lg_new[p.b][is_m]))
    common = old.index.intersection(new.index)
    dr = (np.log(new.loc[common, [p.a, p.b]]).diff() - np.log(old.loc[common, [p.a, p.b]]).diff()).abs()
    rows.append({
        "pair": p.name,
        "g (Study 1 manifest)": man1["hedge_ratios"][p.name],
        "g (new download, IS fit)": g_new,
        "|Δg|": abs(g_new - man1["hedge_ratios"][p.name]),
        "max |Δ log-return| 2015-24": float(dr.max().max()),
    })
integrity = pd.DataFrame(rows).set_index("pair")
display(integrity.round(6))
print(f"largest hedge-ratio discrepancy: {integrity['|Δg|'].max():.2e}")
print(f"largest daily log-return discrepancy: {integrity['max |Δ log-return| 2015-24'].max():.2e}")

**The past is still the past.** Every in-sample hedge ratio from the new download matches
Study 1's manifest to 1e-7, and no daily log return in 2015–2024 differs by more than
1.4e-6 — floating-point residue from the dividend back-adjustment, exactly as the plan
predicted. Yahoo did not revise history between the two downloads, so the 2022–2024 numbers
quoted below are Study 1's numbers, not a re-derivation of them.

## 2. Arm A on the holdout: the replication

Study 1's strategy, unchanged, on twenty months it never saw. Sharpe with Lo's SE, the
stationary-bootstrap interval, and Holm-adjusted p across the ten pairs — exactly the
columns of notebook 03 §5b — plus the expected maximum of ten null Sharpes for a window
this short.

In [ ]:
A_H = oos_inf(iH)
T_H = int(A_H.n_periods.iloc[0])
null_max_H = expected_max_sharpe(len(PAIRS), sharpe_null_sd(T_H))

show = A_H[["total_return", "total_return_ci_lo", "total_return_ci_hi", "sharpe", "sharpe_se",
            "sharpe_ci_lo", "sharpe_ci_hi", "sharpe_p", "sharpe_p_holm"]].copy()
show.columns = ["net ret", "ret CI lo", "ret CI hi", "Sharpe", "SE", "boot lo", "boot hi", "p", "p Holm"]
display(show.sort_values("net ret", ascending=False).round(3))

r = A_H.total_return * 100
t, pval = stats.ttest_1samp(r, 0.0)
print(f"holdout length: {T_H} trading days   ->   SE on a null Sharpe {sharpe_null_sd(T_H):.2f}, "
      f"expected best-of-10 {null_max_H:.2f}")
print(f"net total return: mean {r.mean():+.2f}%  median {r.median():+.2f}%  sd {r.std():.2f}%  "
      f"profitable {int((r > 0).sum())}/10   t = {t:.2f}, p = {pval:.3f}")
print(f"mean net Sharpe {A_H.sharpe.mean():+.2f};  best {A_H.sharpe.idxmax()} = {A_H.sharpe.max():.2f}")
print(f"Holm p < 0.05: {sorted(A_H.index[A_H.sharpe_p_holm < 0.05])}")

**Study 1's strategy lost money on the holdout, and nothing about it is significant.** Two
of ten pairs are profitable (UPS/FDX +24.4%, FOXA/FOX +5.9%); the cross-sectional mean is
−3.5% (t = −0.96, p = 0.36), the median −6.7%. No pair's Holm-adjusted p is below 1. The
best Sharpe, FOXA/FOX at 1.24, sits *exactly* on the expected best-of-ten under no edge
for a window this short (1.23) — the noise floor, hit on the nose.

Two things are worth separating. The cross-sectional standard deviation fell from 26pp to
12pp, which is what twenty months instead of thirty-six does to total returns; it is not
evidence that pairs became more alike. And the one large positive number, UPS/FDX, is the
pair whose relationship Study 1 found to have broken — its holdout interval is
[−13%, +82%], which is a description of a coin, not of an edge.

The pre-stated power caveat applies: with SE 0.78 per Sharpe this window could only have
detected a large effect. It did not detect one, in either direction.

## 3. Persistence: do the 2022–2024 winners win again?

The pre-registered test of Study 1's thesis. If pair outcomes are noise, the rank of a
pair's 2022–2024 return should say nothing about its 2025–2026 return: Spearman ρ ≈ 0, and
about half of the four previous winners repeat. If the ranking replicates, the thesis is
wrong and this section says so.

In [ ]:
prev = oos_net(m1) * 100
now = oos_net(mH) * 100
rho, p_rho = stats.spearmanr(prev, now)
r_p, p_p = stats.pearsonr(prev, now)
winners_prev = sorted(prev.index[prev > 0])
repeat = sorted(p for p in winners_prev if now[p] > 0)
losers_prev = sorted(prev.index[prev <= 0])
stay_losers = sorted(p for p in losers_prev if now[p] <= 0)

print(f"Spearman rho = {rho:+.3f} (p = {p_rho:.3f})   Pearson r = {r_p:+.3f} (p = {p_p:.3f})")
print(f"2022-24 winners: {winners_prev}")
print(f"  ...profitable again in 2025-26: {repeat}  ({len(repeat)} of {len(winners_prev)})")
print(f"2022-24 losers still losing: {len(stay_losers)} of {len(losers_prev)}")

fig, ax = plt.subplots(figsize=(6, 5))
ax.axhline(0, lw=0.8, color="0.7"); ax.axvline(0, lw=0.8, color="0.7")
ax.scatter(prev, now, s=45, zorder=3)
for name in PAIRS:
    ax.annotate(name, (prev[name], now[name]), fontsize=8, xytext=(4, 4), textcoords="offset points")
ax.set_xlabel("2022–2024 net total return %  (Study 1)")
ax.set_ylabel("2025–2026 net total return %  (holdout)")
ax.set_title(f"Persistence of pair outcomes, Arm A  (Spearman ρ = {rho:+.2f})")
fig.tight_layout()

**Zero of the four 2022–2024 winners repeated.** KO/PEP, HD/LOW, UNP/CSX and DUK/SO — the
pairs a reader of Study 1 would have kept — returned −16.7%, −10.7%, −11.5% and −3.0%.
UNP/CSX, the +55.9% headline pair, is now a −11.5% pair. The 2022–2024 worst, UPS/FDX at
−42.4%, is the holdout's best at +24.4%. Spearman ρ = −0.56 (p = 0.09), Pearson
r = −0.63 (p = 0.05).

The pre-registered prediction was ρ ≈ 0 and about two of four winners repeating. The
observation is more extreme than the prediction in the direction the thesis points: not
merely no persistence, but a rank *reversal*, on the edge of significance at n = 10. That
last clause matters — a ρ of −0.56 on ten pairs is suggestive, and this notebook will not
claim that pair performance mean-reverts. What it can claim is that the null of "the
ranking carries information" is not supported by anything here: a trader who had selected
the four winners in December 2024 would have lost on all four.

There is a second, quieter lesson in the holdout run's own in-sample column. With the fit
window extended through 2024, UNP/CSX shows **+74% in-sample and −11.5% out-of-sample** —
the classic overfitting signature that Study 1 said was *absent*. It was absent because
Study 1's in-sample period happened not to contain UNP/CSX's lucky years. Once it does, the
signature appears, with no parameter having been fitted to anything. In-sample performance
that consists of a lucky stretch looks identical to in-sample performance that consists of
an edge.

## 4. Arm B: does re-estimating what goes stale help?

Arm B minus Arm A, per pair, on both windows. On 2022–2024 this is a variant of Study 1
(the diagnosis that motivated Arm B was made on that data, so a gain there is suggestive at
best); on the holdout it is a clean test. The cross-sectional mean difference gets a
bootstrap over pairs — ten of them, so read the interval, not the point.

In [ ]:
def compare(mA, mB, iA, iB, label):
    a, b = oos_net(mA) * 100, oos_net(mB) * 100
    ia, ib = oos_inf(iA), oos_inf(iB)
    out = pd.DataFrame({
        "Arm A ret %": a, "Arm B ret %": b, "B − A (pp)": b - a,
        "Arm A Sharpe": ia.sharpe, "Arm B Sharpe": ib.sharpe, "B − A Sharpe": ib.sharpe - ia.sharpe,
    })
    d = (b - a).to_numpy()
    rng = np.random.default_rng(0)
    boots = rng.choice(d, size=(10000, len(d)), replace=True).mean(axis=1)
    lo, hi = np.percentile(boots, [2.5, 97.5])
    print(f"{label}")
    print(f"  mean net return   A {a.mean():+.2f}%   B {b.mean():+.2f}%   "
          f"B−A {d.mean():+.2f}pp  [{lo:+.2f}, {hi:+.2f}]  (bootstrap over pairs)")
    print(f"  mean net Sharpe   A {ia.sharpe.mean():+.2f}   B {ib.sharpe.mean():+.2f}")
    print(f"  profitable        A {int((a > 0).sum())}/10   B {int((b > 0).sum())}/10   "
          f"B better on {int((d > 0).sum())}/10 pairs\n")
    return out

cmp_2224 = compare(m1, m1b, i1, i1b, "2022–2024 (variant of Study 1)")
cmp_hold = compare(mH, mHb, iH, iHb, "2025–2026 (holdout)")
display(cmp_2224.round(2))
display(cmp_hold.round(2))

In [ ]:
seg = pd.concat({"2022–2024": s1b, "holdout": sHb}, names=["evaluation"]).reset_index(level=0)
seg["half_life"] = seg["half_life"].round(1)
seg["half_life_se"] = seg["half_life_se"].round(1)
seg["hedge_ratio"] = seg["hedge_ratio"].round(3)
display(seg.set_index(["evaluation", "pair", "segment"])[
    ["start", "fit_end", "hedge_ratio", "half_life", "half_life_se", "window", "traded"]
])
print(f"segments not traded (non-positive refit hedge): "
      f"{int((~seg.traded.astype(bool)).sum())} of {len(seg)}")
print(f"windows chosen: {sorted(int(w) for w in seg.window.unique())}")

**Re-estimating what goes stale did not produce a detectable improvement.**

On 2022–2024 — the window whose diagnosis motivated Arm B — the variant is *worse*: mean
−7.2% against +0.6%, better on only three pairs. The pre-registered rule replaced the
60-day window with 80–250 days for nine of ten pairs (SPY/VOO, half-life one day, gets the
floor of 20; MA/V, half-life ~30, is the only pair for which 60 was already about right).
UNP/CSX's +55.9% becomes −10.8% under a 191-day window, which is what notebook 04 §1 had
already found: the headline pair was an artifact of the window. UPS/FDX improves by 38
points for the mirror-image reason — a 250-day window trades it less, and it loses less.

On the holdout the variant is better by 2.9 points (−0.7% against −3.5%), profitable on
five pairs instead of two, better on six of ten — and the bootstrap interval on the mean
difference is [−10, +14]. The per-pair differences are the tell: KO/PEP +38 points,
UPS/FDX −46, UNP/CSX +15, HD/LOW +14, FOXA/FOX −5. A specification that moves individual
pairs by ±45 points and the mean by 3 is not a better estimator; it is a different draw.
This is the same reshuffling signature notebook 04 §5 found for the rolling-versus-static
hedge, and it is what "improvement" looks like when there is no edge underneath to improve.

The segment table shows why the refit itself did so little. On an anchored expanding
window the hedge ratio moves slowly by construction (UPS/FDX 0.84 → 0.97 → 0.88 → 0.78
over four refits; no refit went non-positive, so the flat rule never fired). The active
ingredient was the window rule, and the half-lives it fed on were themselves drifting:
WM/RSG 48 → 126 days, FOXA/FOX 42 → 93, KO/PEP 41 → 291 by the 2026 refit. The speed of
reversion is not a stable property of these pairs either — so a rule that tracks it is
tracking noise with a lag.

## 5. What the holdout established

Everything here was specified in `PREREGISTRATION.md` before the 2025–2026 prices were
downloaded, and everything specified there is reported.

**The replication.** Study 1's frozen strategy, applied to twenty unseen months: 2 of 10
profitable, mean −3.5% (p = 0.36), no pair significant, best Sharpe exactly at the
expected best-of-ten under no edge. Study 1's *numbers* did not replicate. Its *thesis* —
that pair-to-pair dispersion is noise and the ranking carries no information — was tested
directly and held: 0 of 4 winners repeated, ρ = −0.56.

**The variant.** The textbook fix for the diagnosed problem — nothing stays stale, the
window follows the half-life — was worse on the window that motivated it (−7.8 points) and
indistinguishable from the original on the holdout (+2.9, interval [−10, +14]), while
moving individual pairs by up to 46 points. A methodology change that is a-priori
defensible, pre-registered, and implemented without a leak still cannot rescue a strategy
that has no stable edge to begin with. That is not a failure of the fix. It is the answer
to the question the fix was asked.

**What this does not establish.** That pairs trading cannot work — twenty months, one
regime (including the April 2025 tariff shock), ten survivorship-selected large-cap pairs,
SE 0.78 per Sharpe. That pair performance *reverses* — ρ = −0.56 at n = 10 has p = 0.09.
That Arm B is useless in general — on a universe with a real edge, re-estimation might
matter; this universe gave it nothing to work with.

**What it does establish**, and what is worth taking to the next project: a backtest's
headline pair is the maximum of N noisy draws; the in-sample/out-of-sample split protects
against fitted parameters, not against a lucky in-sample period; and the honest test of a
methodological improvement is a pre-registered holdout, which here returned "no detectable
difference" — the most common answer, and the one a study has to be built to be able to
hear.